# 01 - Generate Synthetic Data: Hyper-Personalization Use Case

This notebook generates synthetic shopping/purchase history for 5 users with distinct personas.
Each user has unique purchasing patterns that will be used to train personalized ML models.

**Users:**
- **Alice** - Electronics enthusiast
- **Bob** - Fitness fanatic
- **Carol** - Home chef
- **Dave** - Bookworm
- **Eve** - Fashion forward

**Target Variable:** `repeat_purchase` (0/1) - whether the user will repurchase the item

In [0]:
# Parameters
dbutils.widgets.text("catalog", "custom_ml", "Catalog")  # Only hardcoded default
dbutils.widgets.text("schema", "hyper_personalization", "Schema")
dbutils.widgets.text("num_records_per_user", "200", "Records Per User")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
NUM_RECORDS = int(dbutils.widgets.get("num_records_per_user"))

# Validate catalog exists (do NOT attempt to create it)
catalogs = [row.catalog for row in spark.sql("SHOW CATALOGS").collect()]
assert CATALOG in catalogs, f"Catalog '{CATALOG}' does not exist. Please create it manually or use an existing catalog."

# Create schema if needed
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.{SCHEMA}")
print(f"Schema ready: {CATALOG}.{SCHEMA}")
print(f"Records per user: {NUM_RECORDS}")

## Define User Personas
Each user has a distinct set of products, categories, price ranges, and purchasing behavior.

In [0]:
import numpy as np
import random
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType, DateType

random.seed(42)
np.random.seed(42)

# Define user personas with products, categories, price ranges, and repeat tendency
USER_PERSONAS = {
    "user_alice": {
        "name": "Alice",
        "category": "Electronics",
        "products": [
            ("Laptop", 899.99, 1299.99),
            ("Wireless Headphones", 49.99, 299.99),
            ("Tablet", 329.99, 799.99),
            ("Smartwatch", 199.99, 449.99),
            ("Digital Camera", 399.99, 899.99),
            ("USB-C Charger", 19.99, 59.99),
            ("4K Monitor", 299.99, 699.99),
            ("Mechanical Keyboard", 79.99, 199.99),
        ],
        "repeat_base_rate": 0.65,  # High repeat for electronics
    },
    "user_bob": {
        "name": "Bob",
        "category": "Fitness",
        "products": [
            ("Protein Powder", 29.99, 59.99),
            ("Running Shoes", 89.99, 179.99),
            ("Yoga Mat", 19.99, 69.99),
            ("Dumbbells Set", 49.99, 199.99),
            ("Fitness Tracker", 99.99, 249.99),
            ("Resistance Bands", 14.99, 39.99),
            ("Pre-Workout Supplement", 24.99, 49.99),
        ],
        "repeat_base_rate": 0.70,  # Very high repeat for consumables
    },
    "user_carol": {
        "name": "Carol",
        "category": "Kitchen",
        "products": [
            ("Cast Iron Skillet", 29.99, 79.99),
            ("Spice Set", 19.99, 49.99),
            ("Kitchen Gadget Set", 24.99, 89.99),
            ("Recipe Book", 14.99, 39.99),
            ("Professional Blender", 79.99, 299.99),
            ("Cutting Board Set", 24.99, 69.99),
            ("Chef Knife", 49.99, 199.99),
        ],
        "repeat_base_rate": 0.60,
    },
    "user_dave": {
        "name": "Dave",
        "category": "Books & Reading",
        "products": [
            ("Bestseller Novel", 9.99, 24.99),
            ("E-Reader", 99.99, 249.99),
            ("Leather Bookmark Set", 9.99, 19.99),
            ("Reading Lamp", 29.99, 79.99),
            ("Audiobook Subscription", 14.99, 14.99),
            ("Adjustable Book Stand", 19.99, 49.99),
            ("Premium Notebook", 12.99, 29.99),
        ],
        "repeat_base_rate": 0.75,  # Highest repeat - books are consumable
    },
    "user_eve": {
        "name": "Eve",
        "category": "Fashion",
        "products": [
            ("Designer Handbag", 199.99, 899.99),
            ("Designer Shoes", 149.99, 599.99),
            ("Luxury Sunglasses", 99.99, 399.99),
            ("Designer Watch", 299.99, 999.99),
            ("Gold Jewelry", 149.99, 499.99),
            ("Silk Scarf", 49.99, 199.99),
            ("Premium Perfume", 79.99, 249.99),
        ],
        "repeat_base_rate": 0.50,  # Lower repeat - fashion is more varied
    },
}

print(f"Defined {len(USER_PERSONAS)} user personas")

## Generate Purchase Records
For each user, generate ~200 purchase records with realistic patterns:
- Higher-rated items have higher repeat purchase probability
- Price affects quantity (cheaper items bought in higher quantities)
- Temporal patterns over the past year

In [0]:
def generate_user_data(user_id, persona, num_records=200):
    """Generate synthetic purchase records for a user."""
    records = []
    start_date = datetime(2025, 1, 1)
    end_date = datetime(2026, 5, 1)
    date_range_days = (end_date - start_date).days
    
    for _ in range(num_records):
        # Pick a random product
        product_name, min_price, max_price = random.choice(persona["products"])
        
        # Generate price with some variance
        price = round(random.uniform(min_price, max_price), 2)
        
        # Quantity inversely related to price (cheaper items bought more)
        if price < 30:
            quantity = random.choices([1, 2, 3, 4], weights=[0.2, 0.3, 0.3, 0.2])[0]
        elif price < 100:
            quantity = random.choices([1, 2, 3], weights=[0.4, 0.4, 0.2])[0]
        else:
            quantity = random.choices([1, 2], weights=[0.8, 0.2])[0]
        
        # Random purchase date
        days_offset = random.randint(0, date_range_days)
        purchase_date = start_date + timedelta(days=days_offset)
        
        # Rating: slight bias toward higher ratings (users buy what they like)
        rating = random.choices([1, 2, 3, 4, 5], weights=[0.05, 0.10, 0.20, 0.35, 0.30])[0]
        
        # Repeat purchase probability: base rate + rating bonus + price factor
        repeat_prob = persona["repeat_base_rate"]
        repeat_prob += (rating - 3) * 0.08  # Higher rating = more likely to repeat
        repeat_prob -= (price / max_price) * 0.1  # Expensive items slightly less repeated
        repeat_prob += quantity * 0.03  # Buying more = signals satisfaction
        repeat_prob = np.clip(repeat_prob, 0.1, 0.95)  # Keep in reasonable bounds
        
        repeat_purchase = 1 if random.random() < repeat_prob else 0
        
        records.append((
            user_id,
            product_name,
            persona["category"],
            price,
            quantity,
            purchase_date.date(),
            rating,
            repeat_purchase
        ))
    
    return records

print("Data generation function defined")

In [0]:
# Define schema for purchase records
purchase_schema = StructType([
    StructField("user_id", StringType(), False),
    StructField("product_name", StringType(), False),
    StructField("category", StringType(), False),
    StructField("price", FloatType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("purchase_date", DateType(), False),
    StructField("rating", IntegerType(), False),
    StructField("repeat_purchase", IntegerType(), False),
])

# Generate data for all users and save individual tables
all_records = []

for user_id, persona in USER_PERSONAS.items():
    records = generate_user_data(user_id, persona, num_records=NUM_RECORDS)
    all_records.extend(records)
    
    # Create DataFrame and save as individual user table
    user_df = spark.createDataFrame(records, schema=purchase_schema)
    table_name = f"{CATALOG}.{SCHEMA}.{user_id}_purchases"
    user_df.write.mode("overwrite").saveAsTable(table_name)
    
    # Stats
    repeat_rate = sum(1 for r in records if r[7] == 1) / len(records)
    print(f"  {user_id}: {len(records)} records, repeat_rate={repeat_rate:.2f} -> {table_name}")

print(f"\nTotal records generated: {len(all_records)}")

In [0]:
# Save combined table
combined_df = spark.createDataFrame(all_records, schema=purchase_schema)
combined_table = f"{CATALOG}.{SCHEMA}.all_user_purchases"
combined_df.write.mode("overwrite").saveAsTable(combined_table)
print(f"Combined table saved: {combined_table} ({combined_df.count()} rows)")

In [0]:
# Verify: show sample from each user
print("=" * 80)
print("SAMPLE DATA VERIFICATION")
print("=" * 80)
display(spark.table(f"{CATALOG}.{SCHEMA}.all_user_purchases").orderBy("user_id", "purchase_date").limit(20))